In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import yaml
from dataclasses import dataclass

In [ ]:
DATA_CONFIGURATION = yaml.safe_load(Path("/workspaces/WebIdentification/cv_webidentification.yaml").read_text())
ROOT_DIR = Path("/workspaces/WebIdentification")
DATA_DIR = ROOT_DIR / DATA_CONFIGURATION["path"]
SPLITS = ["train", "val", "test"]


# Check for BBOX variation,
- Place in image
- Size in image
- Class distribution
- Average bbox per image
- Corruputed images
    - None normalized images
    - "Duplicate" labels

In [ ]:
@dataclass
class BBox:
    class_id: int
    x_center: float
    y_center: float
    width: float
    height: float

from collections import Counter, defaultdict
import hashlib

def hash_label_line(line: str) -> str:
    return hashlib.blake2b(line.encode("utf-8"), digest_size=16).hexdigest()

num_labels = {}
label_hash = {}
bboxs = {}
line_hash_to_images = {}

for split in SPLITS:
    label_paths = list((DATA_DIR / split).glob("*.txt"))
    num_labels[split] = []
    label_hash[split] = set()
    bboxs[split] = []
    line_hash_to_images[split] = defaultdict(set)

    duplicate_lines_within_image = 0

    for label_path in label_paths:
        with open(label_path) as f:
            lines = [line.strip() for line in f if line.strip()]

        file_line_hashes = [hash_label_line(line) for line in lines]
        hashed_lines = hash(tuple(file_line_hashes))
        label_hash[split].add(hashed_lines)

        num_labels[split].append(len(lines))

        # Detect duplicate lines inside the same image label file.
        local_counts = Counter(file_line_hashes)
        if any(count > 1 for count in local_counts.values()):
            duplicate_lines_within_image += 1
            print(f"Duplicate label line(s) inside file: {label_path}")

        # Track where each line hash appears to detect duplicates across images.
        for line_hash in file_line_hashes:
            line_hash_to_images[split][line_hash].add(label_path.name)

        for line in lines:
            class_id, x_center, y_center, width, height = line.split()
            bboxs[split].append(BBox(
                int(class_id),
                float(x_center),
                float(y_center),
                float(width),
                float(height)
                )
            )

        if len(lines) == 0:
            print(f"Corrupted image found: {label_path.with_suffix('.png')}")

    average_bbox_per_image = sum(num_labels[split]) / len(label_paths) if label_paths else 0.0
    print(f"Average bbox per image for {split}: {average_bbox_per_image:.3f}")

    duplicate_label_files = len(label_paths) - len(label_hash[split])
    print(f"Duplicate full label files for {split}: {duplicate_label_files} (average {duplicate_label_files / len(label_paths):.3f})" if label_paths else f"Duplicate full label files for {split}: 0")

    line_hashes_reused_across_images = {
        line_hash: image_names
        for line_hash, image_names in line_hash_to_images[split].items()
        if len(image_names) > 1
    }
    print(f"Files with duplicate lines inside image for {split}: {duplicate_lines_within_image}")
    print(f"Line hashes reused across different images for {split}: {len(line_hashes_reused_across_images)}")

In [ ]:
for split in SPLITS:
    plt.figure(figsize=(10, 6))
    sns.histplot(num_labels[split], bins=20, kde=True)
    plt.title(f"Distribution of number of labels per image for {split} split")
    plt.xlabel("Number of labels")
    plt.ylabel("Frequency")
    plt.show()


In [ ]:
for split in SPLITS:
    x_centers = [bbox.x_center for bbox in bboxs[split]]
    y_centers = [bbox.y_center for bbox in bboxs[split]]

    plt.figure(figsize=(10, 6))
    sns.histplot(
        x=x_centers,
        y=y_centers,
        bins=40,
        cbar=True,
        cmap="viridis"
    )
    plt.title(f"Heatmap of bbox centers for {split} split")
    plt.xlabel("X Center")
    plt.ylabel("Y Center")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.show()


In [ ]:
import numpy as np
import matplotlib.colors as mcolors

# Fixed analysis canvas (width x height)
IMG_W = 1920
IMG_H = 1080

for split in SPLITS:
    # uint16 is often enough for overlap counts and keeps memory low (~4 MB/grid).
    overlap = np.zeros((IMG_H, IMG_W), dtype=np.uint16)

    for bbox in bboxs[split]:
        # Center each bbox at (0.5, 0.5) to analyze size distribution only.
        x_center = 0.5
        y_center = 0.5

        left = int((x_center - bbox.width / 2) * IMG_W)
        right = int((x_center + bbox.width / 2) * IMG_W)
        top = int((y_center - bbox.height / 2) * IMG_H)
        bottom = int((y_center + bbox.height / 2) * IMG_H)

        # Clamp to canvas bounds and skip empty/out-of-frame boxes.
        left = max(0, min(IMG_W, left))
        right = max(0, min(IMG_W, right))
        top = max(0, min(IMG_H, top))
        bottom = max(0, min(IMG_H, bottom))

        if right <= left or bottom <= top:
            continue

        overlap[top:bottom, left:right] += 1

    fig, ax = plt.subplots(figsize=(12, 7))
    vmax = int(overlap.max())

    if vmax > 0:
        # Log scale helps reveal both dense center and sparse outer regions.
        norm = mcolors.LogNorm(vmin=1, vmax=vmax)
        im = ax.imshow(overlap, cmap="magma", origin="lower", norm=norm)
        cbar_label = "BBox overlap count (log scale)"
    else:
        im = ax.imshow(overlap, cmap="magma", origin="lower")
        cbar_label = "BBox overlap count"

    fig.colorbar(im, ax=ax, label=cbar_label)
    ax.set_title(f"Centered bbox size overlap heatmap for {split} ({IMG_W}x{IMG_H})")
    ax.set_xlabel("Pixel X")
    ax.set_ylabel("Pixel Y")
    plt.tight_layout()
    plt.show()

In [ ]:
zero_zero_counts = {}
total_zero_zero = 0

for split in SPLITS:
    count = sum(1 for bbox in bboxs[split] if bbox.width == 0.0 and bbox.height == 0.0)
    zero_zero_counts[split] = count
    total_zero_zero += count
    print(f"0x0 boxes in {split}: {count}")

print(f"Total 0x0 boxes across all splits: {total_zero_zero}")

In [ ]:
zero_zero_counts = {}
total_zero_zero = 0

for split in SPLITS:
    count = sum(1 for bbox in bboxs[split] if bbox.x_center < 0.1 and bbox.y_center < 0.1)
    zero_zero_counts[split] = count
    total_zero_zero += count
    print(f"0x0 boxes in {split}: {count}")

print(f"Total 0x0 boxes across all splits: {total_zero_zero}")